# Cross-modal Inference with PE-AV

[`facebook/pe-av-base-16-frame`](https://huggingface.co/facebook/pe-av-base-16-frame) is Meta's **Perception Encoder for Audio-Video**: a multimodal encoder that embeds audio, video, audio+video, and text into a shared space trained with contrastive learning.

In this notebook we'll:
1. Load the model and processor from the Hub.
2. Download a few sample video clips.
3. Run **zero-shot video classification** and **text → video retrieval**.
4. Stream the [`OpenSound/AudioCaps`](https://huggingface.co/datasets/OpenSound/AudioCaps) validation split and measure **audio ↔ text retrieval** (Recall@1/5/10).

We use the `-16-frame` variant: it subsamples 16 frames per clip, which keeps memory well within a 16 GB Colab GPU. The full-frame `facebook/pe-av-base` follows the same API but is significantly heavier.

## Setup

In [1]:
# !pip install -q -U transformers datasets huggingface_hub torchcodec av soundfile librosa timm

In [2]:
import torch
from transformers import PeAudioVideoModel, PeAudioVideoProcessor

model_id = "facebook/pe-av-base-16-frame"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = PeAudioVideoProcessor.from_pretrained(model_id)
model = PeAudioVideoModel.from_pretrained(model_id).to(device).eval()

print(f"Loaded {model_id} on {device}")

/opt/conda/envs/test-pe/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `PeAudioVideoProcessor` defines `feature_extractor_class = 'PeAudioFeatureExtractor'`, which is deprecated. Register the correct mapping in `AutoFeatureExtractor` instead.
[transformers] `PeAudioVideoProcessor` defines `video_processor_class = 'PeVideoVideoProcessor'`, which is deprecated. Register the correct mapping in `AutoVideoProcessor` instead.


[ERROR] `padding_mask_videos` is part of PeVideoModel.__init__.<locals>.get_video_features's signature, but not documented. Make sure to add it to the docstring of the function in /opt/conda/envs/test-pe/lib/python3.12/site-packages/transformers/models/pe_video/modeling_pe_video.py.


Loading weights: 100%|██████████| 903/903 [00:00<00:00, 4816.61it/s]


Loaded facebook/pe-av-base-16-frame on cuda


## Encoding helpers

The joint `model(**inputs)` forward expects matched batch sizes across modalities, which is inconvenient for retrieval and classification. We drive the video, text, and audio sub-encoders directly — this mirrors what the joint forward does internally and lets us encode any number of videos against any number of text queries.

In [3]:
def encode_videos(paths):
    inputs = processor(videos=paths, return_tensors="pt").to(device)
    with torch.inference_mode(), torch.autocast(device.type, dtype=torch.bfloat16):
        out = model.video_model.video_encoder(pixel_values_videos=inputs["pixel_values_videos"])
        return model.video_model.video_head(out.pooler_output)

def encode_text_for_video(texts):
    inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)
    with torch.inference_mode(), torch.autocast(device.type, dtype=torch.bfloat16):
        out = model.video_model.text_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_hidden_states=True,
        )
        cls = out.hidden_states[-1][:, 0]  # CLS token
        return model.video_model.text_video_head(cls)

AUDIO_SR = processor.feature_extractor.sampling_rate  # 48 kHz

def encode_audio(audio_inputs, sampling_rate=AUDIO_SR):
    """audio_inputs: list of file paths OR list of 1-D numpy waveforms already at `sampling_rate`."""
    inputs = processor(audio=audio_inputs, sampling_rate=sampling_rate, return_tensors="pt").to(device)
    with torch.inference_mode(), torch.autocast(device.type, dtype=torch.bfloat16):
        out = model.audio_model.audio_encoder(input_values=inputs["input_values"])
        return model.audio_model.audio_head(out.pooler_output)

def encode_text_for_audio(texts):
    inputs = processor(text=texts, return_tensors="pt", padding=True).to(device)
    with torch.inference_mode(), torch.autocast(device.type, dtype=torch.bfloat16):
        out = model.audio_model.text_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_hidden_states=True,
        )
        cls = out.hidden_states[-1][:, 0]
        return model.audio_model.text_audio_head(cls)

## Grab a few sample videos

We pull a handful of short clips from a public Hub dataset so the notebook is self-contained. Swap these paths for your own `.mp4` files if you prefer.

In [4]:
from huggingface_hub import hf_hub_download

VIDEO_REPO = "raushan-testing-hf/videos-test"
video_names = ["sample_demo_1.mp4", "karate.mp4", "Cooking_cake.mp4"]

video_files = [
    hf_hub_download(repo_id=VIDEO_REPO, filename=name, repo_type="dataset")
    for name in video_names
]
video_files

['/home/steven_huggingface_co/.cache/huggingface/hub/datasets--raushan-testing-hf--videos-test/snapshots/4cba700bd771f44d72b549253da025c32e944d42/sample_demo_1.mp4',
 '/home/steven_huggingface_co/.cache/huggingface/hub/datasets--raushan-testing-hf--videos-test/snapshots/4cba700bd771f44d72b549253da025c32e944d42/karate.mp4',
 '/home/steven_huggingface_co/.cache/huggingface/hub/datasets--raushan-testing-hf--videos-test/snapshots/4cba700bd771f44d72b549253da025c32e944d42/Cooking_cake.mp4']

## Zero-shot video classification

PE-AV aligns video and text in a shared embedding space, so we can score any video against a list of candidate prompts — no fine-tuning needed.

In [5]:
import os

candidate_labels = [
    "a person practicing karate",
    "someone baking a cake in a kitchen",
    "a dog playing in the park",
    "people dancing at a party",
    "a cat sleeping on a couch",
    "someone playing the guitar",
]

video_embeds = encode_videos(video_files)
text_embeds = encode_text_for_video(candidate_labels)

sim = video_embeds.float() @ text_embeds.float().T  # (num_videos, num_labels)
probs = sim.softmax(dim=-1).cpu()

for path, row in zip(video_files, probs):
    print(f"\n{os.path.basename(path)}")
    top = torch.topk(row, k=3)
    for score, idx in zip(top.values, top.indices):
        print(f"  {score.item():.3f}  {candidate_labels[idx]}")


sample_demo_1.mp4
  0.247  a person practicing karate
  0.173  someone playing the guitar
  0.167  people dancing at a party

karate.mp4
  0.551  a person practicing karate
  0.116  someone playing the guitar
  0.089  a dog playing in the park

Cooking_cake.mp4
  0.576  someone baking a cake in a kitchen
  0.119  people dancing at a party
  0.106  a person practicing karate


Expected top-1: `karate.mp4 → practicing karate`, `Cooking_cake.mp4 → baking a cake`.

## Text → video retrieval

Same embeddings, different reading: for each query we rank the videos in our tiny gallery.

In [6]:
queries = [
    "martial arts training",
    "baking dessert",
]

query_embeds = encode_text_for_video(queries)
ranking = (query_embeds.float() @ video_embeds.float().T).cpu()

for query, row in zip(queries, ranking):
    order = row.argsort(descending=True)
    print(f"\nQuery: {query!r}")
    for rank, idx in enumerate(order, start=1):
        print(f"  {rank}. {os.path.basename(video_files[idx])}  (sim={row[idx].item():.3f})")


Query: 'martial arts training'
  1. karate.mp4  (sim=1.739)
  2. sample_demo_1.mp4  (sim=0.248)
  3. Cooking_cake.mp4  (sim=-0.351)

Query: 'baking dessert'
  1. Cooking_cake.mp4  (sim=1.476)
  2. sample_demo_1.mp4  (sim=-0.199)
  3. karate.mp4  (sim=-0.297)


## Audio ↔ text retrieval on AudioCaps

Now the audio side of the model. We'll stream a few dozen clips from the [`OpenSound/AudioCaps`](https://huggingface.co/datasets/OpenSound/AudioCaps) validation split — each example is a 10 s YouTube clip with a human-written caption — embed both modalities, and measure retrieval accuracy.

In [7]:
import numpy as np
import librosa
from datasets import load_dataset

N = 64  # number of validation clips to embed

ds = load_dataset("OpenSound/AudioCaps", split="validation", streaming=True)
waves, captions = [], []
for i, ex in enumerate(ds):
    if i >= N:
        break
    samples = ex["audio"].get_all_samples()
    wav = samples.data.numpy()
    if wav.ndim > 1:
        wav = wav.mean(axis=0)  # to mono
    sr = samples.sample_rate
    if sr != AUDIO_SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=AUDIO_SR)
    waves.append(wav.astype(np.float32))
    captions.append(ex["caption"])

print(f"Loaded {len(waves)} clips — first clip: {len(waves[0]) / AUDIO_SR:.1f}s at {AUDIO_SR} Hz")

Loaded 64 clips — first clip: 10.0s at 48000 Hz


In [8]:
def batched(fn, items, batch_size):
    return torch.cat([fn(items[i:i + batch_size]) for i in range(0, len(items), batch_size)])

audio_embeds = batched(encode_audio, waves, batch_size=8)
text_embeds = batched(encode_text_for_audio, captions, batch_size=32)

audio_embeds.shape, text_embeds.shape

(torch.Size([64, 1024]), torch.Size([64, 1024]))

Retrieval metrics. For each audio clip, the ground-truth caption is the one at the same index; we rank all captions by similarity and check whether the correct one falls in the top-k.

In [9]:
sim = (audio_embeds.float() @ text_embeds.float().T).cpu()  # (N, N)

def recall_at_k(scores, k):
    ranks = scores.argsort(dim=-1, descending=True)
    gt = torch.arange(scores.shape[0]).unsqueeze(1)
    return (ranks[:, :k] == gt).any(dim=-1).float().mean().item()

print(f"Clips: {sim.shape[0]}")
for k in (1, 5, 10):
    print(f"  Audio → Text  R@{k}: {recall_at_k(sim,   k):.3f}")
for k in (1, 5, 10):
    print(f"  Text  → Audio R@{k}: {recall_at_k(sim.T, k):.3f}")

Clips: 64
  Audio → Text  R@1: 0.766
  Audio → Text  R@5: 0.984
  Audio → Text  R@10: 1.000
  Text  → Audio R@1: 0.750
  Text  → Audio R@5: 1.000
  Text  → Audio R@10: 1.000


A few qualitative audio→text retrievals — for each audio query we print the top-3 captions, with `*` marking the ground-truth match.

In [10]:
for i in range(5):
    order = sim[i].argsort(descending=True)[:3].tolist()
    print(f"\n[{i}] GT: {captions[i]}")
    for rank, j in enumerate(order, start=1):
        mark = "*" if j == i else " "
        print(f"  {rank}{mark} ({sim[i, j].item():+.2f}) {captions[j]}")


[0] GT: Rustling occurs, ducks quack and water splashes, followed by an adult female and adult male speaking and duck calls being blown
  1* (+1.45) Rustling occurs, ducks quack and water splashes, followed by an adult female and adult male speaking and duck calls being blown
  2  (+1.28) A woman speaks happily and an animal chirps
  3  (+1.26) A male is speaking and a duck quacks as others laugh

[1] GT: An audience gives applause as a man yells and a group sings
  1* (+1.70) An audience gives applause as a man yells and a group sings
  2  (+0.33) Some tunes played by whistling
  3  (+0.30) A sheep baa followed by birds chirping and then more sheep baaing

[2] GT: A man speaks over intermittent keyboard taps
  1* (+1.35) A man speaks over intermittent keyboard taps
  2  (+0.70) Several insects fly while two men talk
  3  (+0.69) Someone is typing on a computer keyboard

[3] GT: Motor noise is followed by a horn honking and a siren wailing
  1* (+1.82) Motor noise is followed by a hor

## Where to go next

- **Build a retrieval index**: run `encode_videos` over a larger corpus and serve nearest-neighbor search with FAISS or usearch.
- **Try the bigger checkpoints**: `facebook/pe-av-large-16-frame` pushes retrieval scores higher at the same frame count.
- **Combine modalities**: the model also exposes `audio_video_encoder` for joint audio+video embeddings — useful when the soundtrack carries signal (music, speech, ambient scene).